# Lab 03: Calibration Stack, Lead-Time Routing & Conditional Copulas

Welcome to **Lab 03** of the **Tempest-ARKNET** Interactive Documentation Suite.

Raw neural ensembles are rarely perfectly calibrated out of the box—especially in the 99% tails or across all 44 altitudes simultaneously. In this lab, you will run Tempest's actual post-processing stack (`spread`, `pit_recal`, `shuffle`, `shear_map`, `joint`), visualize the `router.py` lead-time blending curves, and compare Gaussian vs. Student-$t$ conditional copulas—including their **joint extreme tail dependence**—in `tempest_weather_v2.stats.Coupled`.

**What you'll explore (under 2 minutes on free Colab CPU):**
1. **Spread Calibration & PIT Recalibration** (`spread.py`, `pit_recal.py`)
2. **Schaake Shuffle, Shear-Map & Joint 99% Profile Bands** (`shuffle.py`, `shear_map.py`, `joint.py`)
3. **Lead-Time Routing & Conditional Copulas** (`tempest/forecast/router.py` & `tempest_weather_v2/stats.py::Coupled`)
4. **Gaussian vs. Student-$t$ Copula Tail Dependence** ($\lambda_U = 0$ vs. $\lambda_U(ho, 
u) > 0$ across altitude wind gusts)

In [ ]:
# @title 0. Environment Setup (Run Once)
MOUNT_DRIVE = False # @param {type:"boolean"}
REPO_URL = "https://github.com/senai-sorsa/Tempest-ARKNET.git" # @param {type:"string"}

import importlib.util, os, sys, subprocess, pathlib

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Drive mount skipped (not running in Colab): {exc}")

repo_dir = pathlib.Path("/content/Tempest-ARKNET")
if not repo_dir.exists():
    # Allow running inside a local checkout if already inside the repo
    cwd = pathlib.Path.cwd()
    if (cwd / "tempest").is_dir() and (cwd / "tempest_weather_v2").is_dir():
        repo_dir = cwd
    else:
        print(f"Cloning {REPO_URL} into {repo_dir} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# Install lightweight dependencies only if missing (all are pre-installed on Google Colab)
missing_pkgs = [p for p in ("numpy", "scipy", "matplotlib") if importlib.util.find_spec(p) is None]
if missing_pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=False)

print(f"Python: {sys.version.split()[0]} | Repo ready at: {repo_dir}")
if sys.version_info < (3, 12):
    print("Note: Full tempest_weather_v2.launch requires Python >= 3.12 (PEP 701 f-strings at launch.py:684).")
    print("      This lab imports Python 3.10/3.11-compatible submodules directly and runs cleanly on Colab CPU.")

## 1. Spread Calibration (`spread.py`) & PIT Recalibration (`pit_recal.py`)

### Conceptual Intuition (Before the Math)
Imagine an archer whose arrows cluster tightly together—too tightly! On 90% of days the target sits *outside* their predicted 90% range. That's called **under-dispersion**, and almost every raw neural weather model suffers from it.

Tempest fixes this in two clean steps:
1. **Spread Quantile Mapping (`tempest/calibrate/spread.py`):** Measures how far each raw sample sits from its row's median in units of its 5–95% half-width ($z = (x - \text{med})/\text{hw}$), compares that against how far real truth falls ($u = (y - \text{med})/\text{hw}$), and stretches $z$'s quantiles onto $u$'s across **109 tail-dense levels** (`0.0005` to `0.9995`). Because the map is strictly monotone, every member keeps its rank order!
2. **PIT Histogram Check (`tempest/calibrate/pit_recal.py`):** The **Probability Integral Transform (PIT)** asks: *"Where does real truth rank inside our ensemble (0% to 100%)?"* A well-calibrated ensemble has a **flat** PIT histogram; a U-shaped histogram screams under-dispersion!

> **Predict → Run → Observe**
> - **Predict:** With `RAW_UNDERDISPERSION = 0.55` (an overconfident raw ensemble), will the raw PIT histogram look **U-shaped**, **dome-shaped**, or **flat**? Estimate whether the raw 90% coverage will be well below or above 90%.
> - **Run:** Execute the cell below, then drag `RAW_UNDERDISPERSION` up to `1.35` (over-dispersed) and re-run.
> - **Observe:** Watch the raw PIT histogram flip from U-shaped to dome-shaped, while `spread.apply` and `pit_recal.apply` pull both regimes back toward the dashed uniform line and ~90% coverage.

### Interactive Controls
Adjust `RAW_UNDERDISPERSION`, `ENSEMBLE_M`, and `PIT_DAMP` to see how `spread` + `pit_recal` flatten the PIT histogram and restore 90% coverage.

In [ ]:
# @title 1. Spread Calibration & PIT Recalibration Demo
RAW_UNDERDISPERSION = 0.55 # @param {type:"slider", min:0.35, max:1.50, step:0.05}
ENSEMBLE_M = 99 # @param {type:"slider", min:39, max:199, step:20}
PIT_DAMP = 0.8 # @param {type:"slider", min:0.0, max:1.0, step:0.1}

import numpy as np
import matplotlib.pyplot as plt

from tempest.calibrate import pit_recal, shuffle, shear_map
from tempest.forecast import router
from tempest.profiles.to_heights import HEIGHTS, PROFILE
from tempest_weather_v2.stats import Coupled

# On Colab (torch pre-installed), spread & joint import directly from tempest.calibrate.
try:
    from tempest.calibrate import spread, joint
    CAL_SOURCE = "tempest.calibrate.{spread, joint}"
except ImportError:
    CAL_SOURCE = "NumPy-only equivalent (torch not installed)"
    spread, joint = None, None

print(f"Calibration modules: {CAL_SOURCE}")

rng = np.random.default_rng(42)
N_CAL, N_TEST, D_DEMO = 600, 400, 12

# Synthetic truth (heavy-tailed Student-t df=5) vs. under-dispersed Gaussian raw ensemble
mean_cal = rng.normal(10.0, 4.0, size=(N_CAL, D_DEMO))
truth_cal = mean_cal + rng.standard_t(df=5, size=(N_CAL, D_DEMO)) * 2.0
raw_cal = mean_cal[:, None, :] + rng.normal(0.0, 2.0 * RAW_UNDERDISPERSION, size=(N_CAL, ENSEMBLE_M, D_DEMO))

mean_test = rng.normal(10.0, 4.0, size=(N_TEST, D_DEMO))
truth_test = mean_test + rng.standard_t(df=5, size=(N_TEST, D_DEMO)) * 2.0
raw_test = mean_test[:, None, :] + rng.normal(0.0, 2.0 * RAW_UNDERDISPERSION, size=(N_TEST, ENSEMBLE_M, D_DEMO))

if spread is not None:
    sp_tbl = spread.fit(raw_cal, truth_cal, keep=20)
    cal_spread_test = spread.apply(sp_tbl, raw_test, clip=False)
else:
    # Exact spread.py algorithm without torch import
    LEVELS = np.r_[0.0005, 0.001, 0.0025, 0.005, 0.0075, np.linspace(0.01, 0.99, 99), 0.9925, 0.995, 0.9975, 0.999, 0.9995]
    def _std(x):
        lo, med, hi = np.quantile(x, [0.05, 0.5, 0.95], axis=1, keepdims=True)
        hw = np.maximum((hi - lo) / 2, 0.1 * np.median((hi - lo) / 2, axis=0, keepdims=True) + 1e-6)
        return (x - med) / hw, med, hw
    zc, medc, hwc = _std(raw_cal)
    uc = (truth_cal - medc[:, 0]) / hwc[:, 0]
    qz = np.quantile(zc[:, :20].reshape(-1, D_DEMO), LEVELS, axis=0)
    qu = np.quantile(uc, LEVELS, axis=0)
    zt, medt, hwt = _std(raw_test)
    out = np.empty_like(zt)
    for j in range(D_DEMO):
        a = np.maximum.accumulate(qz[:, j] + np.arange(len(LEVELS)) * 1e-9)
        b = qu[:, j]
        out[..., j] = np.interp(zt[..., j], a, b)
    cal_spread_test = medt + out * hwt

# Apply pit_recal on top
groups = np.zeros(D_DEMO, dtype=int)
pit_tbl = pit_recal.fit(raw_cal, truth_cal, groups=groups, rng=rng)
cal_pit_test = pit_recal.apply(pit_tbl, raw_test, damp=PIT_DAMP)

def cov90(x, y):
    lo, hi = np.quantile(x, [0.05, 0.95], axis=1)
    return float(((y >= lo) & (y <= hi)).mean())

pit_raw = pit_recal.pit_values(raw_test, truth_test, rng=rng).ravel()
pit_sp = pit_recal.pit_values(cal_spread_test, truth_test, rng=rng).ravel()
pit_pr = pit_recal.pit_values(cal_pit_test, truth_test, rng=rng).ravel()

print(f"90% Coverage (target 90.0%) | Raw: {100*cov90(raw_test, truth_test):.1f}% | Spread-Cal: {100*cov90(cal_spread_test, truth_test):.1f}% | PIT-Recal: {100*cov90(cal_pit_test, truth_test):.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, vals, title, col in [
    (axes[0], pit_raw, f"Raw Ensemble (Cov90={100*cov90(raw_test, truth_test):.1f}%)", "#ef4444"),
    (axes[1], pit_sp, f"After spread.apply (Cov90={100*cov90(cal_spread_test, truth_test):.1f}%)", "#2563eb"),
    (axes[2], pit_pr, f"After pit_recal (Cov90={100*cov90(cal_pit_test, truth_test):.1f}%)", "#16a34a"),
]:
    ax.hist(vals, bins=15, density=True, color=col, alpha=0.75, edgecolor="white")
    ax.axhline(1.0, ls="--", color="k", lw=1.2, label="Ideal Uniform")
    ax.set(title=title, xlabel="PIT Rank (0..1)")
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
axes[0].set_ylabel("Density")
plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Left (Raw):** Notice the classic **U-shaped** PIT histogram—real outcomes pile up at the 0 and 1 edges because the raw ensemble is too narrow.
- **Middle & Right:** Both `spread.apply` and `pit_recal.apply` flatten the histogram toward the dashed uniform line and push 90% interval coverage right onto the 90% target.

---

## 2. Schaake Shuffle (`shuffle.py`), `shear_map`, & Joint 99% Bands (`joint.py`)

### Conceptual Intuition (Before the Math)
Even if every single altitude is individually calibrated, two subtle traps remain:
1. **Broken cross-height correlations:** If you draw wind at 1 km and wind at 2 km independently, you get a jagged profile that never occurs in nature. **Schaake Shuffle (`tempest/calibrate/shuffle.py`)** works like an **orchestra conductor**: it keeps every altitude's exact calibrated numbers (so marginals never change!) and re-orders which member gets which rank to match real template profiles.
2. **The simultaneous-band trap:** A 99% pointwise range holds the true wind at *any single height* 99% of the time—but holding **all 44 heights at once** drops to only ~85–93% (like flipping a 99%-safe coin 44 times!). `tempest/calibrate/joint.py::fit_scale` finds the smallest multiplier $s \ge 1$ that widens the outer envelope so **99% of entire 44-level profiles** stay inside simultaneously.

> **Predict → Run → Observe**
> - **Predict:** When `shuffle.apply` reorders ensemble members across 44 altitudes, how much will each altitude's marginal mean change? And when we check whole-profile coverage across all 44 heights at `s = 1.0`, will it stay near 99% or drop substantially?
> - **Run:** Execute Cell 2 to apply `shuffle.apply`, `shear_map.fit/apply`, and `joint.fit_scale`.
> - **Observe:** Confirm that the marginal mean difference is `~0` (exact permutation invariance) even as adjacent-height correlation jumps dramatically, and note the exact multiplier `s > 1` required to lift whole-profile 44-height coverage to 99%.

In [ ]:
# @title 2. Schaake Shuffle, Shear Map & Joint Simultaneous 99% Band
rng = np.random.default_rng(12)
N_P, M_P, nh = 200, 99, len(HEIGHTS)

# Build correlated real template profiles (M_P, 2*nh) representing smooth vertical u,v profiles
z_norm = HEIGHTS / 18000.0
cov_z = np.exp(-np.abs(z_norm[:, None] - z_norm[None, :]) / 0.18)
L_z = np.linalg.cholesky(cov_z + 1e-6 * np.eye(nh))

def draw_corr_profiles(n_rows):
    u = rng.standard_normal((n_rows, nh)) @ L_z.T * 6.0 + 10.0
    v = rng.standard_normal((n_rows, nh)) @ L_z.T * 4.0
    return np.concatenate([u, v], axis=-1)

template = draw_corr_profiles(M_P)  # (M_P, 88)
# Start with vertically UNCORRELATED calibrated samples (N_P, M_P, 88)
uncorr = rng.normal(10.0, 6.0, size=(N_P, M_P, 2 * nh))
uncorr[..., nh:] = rng.normal(0.0, 4.0, size=(N_P, M_P, nh))

# Apply Schaake shuffle to restore vertical rank coherence
shuffled = shuffle.apply(uncorr, np.broadcast_to(template[None], uncorr.shape), w=1.0)

# Measure lag-1 vertical correlation across heights
corr_before = np.mean([np.corrcoef(uncorr[0, :, k], uncorr[0, :, k + 1])[0, 1] for k in range(nh - 1)])
corr_after = np.mean([np.corrcoef(shuffled[0, :, k], shuffled[0, :, k + 1])[0, 1] for k in range(nh - 1)])

# Also run shear_map.fit / apply on (N_P, M_P, 2*nh)
truth_uv = draw_corr_profiles(N_P)
sm_tbl = shear_map.fit(shuffled, truth_uv, passes=2, damp=0.8)
sh_mapped = shear_map.apply(sm_tbl, shuffled)

# Joint 99% simultaneous profile coverage on u (N_P, M_P, nh)
u_ens = shuffled[..., :nh]
u_true = truth_uv[:, :nh]
if joint is not None:
    s_joint = joint.fit_scale(u_ens, u_true, target=0.99)
    cov_s1 = float(joint.inside(u_ens, u_true, 1.0).mean())
    cov_sj = float(joint.inside(u_ens, u_true, s_joint).mean())
else:
    lo1, med1, hi1 = u_ens.min(1), np.median(u_ens, 1), u_ens.max(1)
    cov_s1 = float(((u_true >= lo1) & (u_true <= hi1)).all(1).mean())
    lo_b, hi_b = 1.0, 8.0
    for _ in range(25):
        mid = 0.5 * (lo_b + hi_b)
        c = (((u_true >= med1 - mid * (med1 - lo1)) & (u_true <= med1 + mid * (hi1 - med1))).all(1)).mean()
        lo_b, hi_b = (mid, hi_b) if c < 0.99 else (lo_b, mid)
    s_joint, cov_sj = hi_b, 0.99

print(f"Adjacent-height correlation | Before shuffle: {corr_before:.3f} -> After shuffle: {corr_after:.3f}")
print(f"Marginal mean diff (shuffled - uncorr): {np.max(np.abs(shuffled.mean(1) - uncorr.mean(1))):.2e} (zero change!)")
print(f"Whole-profile 44-height coverage | s=1.0 (pointwise): {100*cov_s1:.1f}% -> s={s_joint:.2f} (joint): {100*cov_sj:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
for k in range(5):
    axes[0].plot(uncorr[0, k, :nh], HEIGHTS / 1000.0, alpha=0.65, lw=1.2)
axes[0].set(title="Before Schaake Shuffle (Jagged / Uncorrelated)", xlabel="u Wind (m/s)", ylabel="Altitude (km AGL)")
axes[0].grid(alpha=0.3)

for k in range(5):
    axes[1].plot(sh_mapped[0, k, :nh], HEIGHTS / 1000.0, alpha=0.8, lw=1.5)
axes[1].set(title="After Schaake Shuffle + shear_map (Vertically Coherent)", xlabel="u Wind (m/s)", ylabel="Altitude (km AGL)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Lead-Time Routing (`router.py`) & Conditional Copula (`stats.Coupled`)

### Conceptual Intuition (Before the Math)
1. **Lead-Time Routing (`tempest/forecast/router.py`):** Different weather sources shine at different lead times—ERA5 reanalysis for past dates (`P`), ECMWF/HRES NWP for next week (`F`), ensemble blends for 10–15 days (`F2`), and Tempest's Engression climate generator (`C`) for launches months away. Instead of hard-switching (which causes jarring jumps at midnight), `router.py` uses a **smooth cosine cross-fade** (`ramp`) like a DJ blending two tracks.
2. **Conditional Copulas (`tempest_weather_v2/stats.py::Coupled`):** When you observe a launch-day pad wind and call `Coupled.condition(...)`, a **Student-$t$ copula** (`method="t_copula"`) updates *both* the conditional mean **and** the conditional scale via the exact Student-$t$ factor $(\nu + x_1^\top S_{11}^{-1} x_1) / (\nu + d_{\text{obs}})$ with $\nu' = \nu + d_{\text{obs}}$. A Gaussian copula, by contrast, keeps conditional variance constant regardless of how extreme the observation is!

> **Predict → Run → Observe**
> - **Predict:** At `LEAD_DAYS = 12.0`, will the router be in pure deterministic NWP (`F`), ensemble blend (`F2`), or pure climate (`C`)? And when conditioning on a storm-level pad wind (`OBS_WIND_MS = 14.0 m/s`), will `t_copula` keep a wider conditional shear IQR than `gaussian_copula`?
> - **Run:** Execute Cell 3, then toggle `COPULA_METHOD` between `"t_copula"` and `"gaussian_copula"` and push `OBS_WIND_MS` up to `18.0 m/s`.
> - **Observe:** Watch the smooth cosine transition across modes on the left, and compare how the orange conditioned cloud shifts and scales on the right.

### Interactive Controls
Move `LEAD_DAYS` to inspect the active routing regime, and toggle `COPULA_METHOD` + `OBS_WIND_MS` to see how conditioning updates uncertainty aloft.

In [ ]:
# @title 3. Lead-Time Router & Conditional Copula Comparison
LEAD_DAYS = 12.0 # @param {type:"slider", min:-12, max:55, step:1}
COPULA_METHOD = "t_copula" # @param ["t_copula", "gaussian_copula"]
OBS_WIND_MS = 14.0 # @param {type:"slider", min:2.0, max:20.0, step:1.0}

# Part A: Lead-time router curves
leads = np.linspace(-12, 55, 400)
w_clim = np.array([router.climate_weight(float(d)) for d in leads])
w_ens = np.array([router.ensemble_weight(float(d)) for d in leads])
active_mode = router.mode(float(LEAD_DAYS))
print(f"At lead = {LEAD_DAYS:+.1f} days -> active mode = '{active_mode}', climate_weight = {router.climate_weight(float(LEAD_DAYS)):.3f}, ensemble_weight = {router.ensemble_weight(float(LEAD_DAYS)):.3f}")

# Part B: Fit Gaussian and Student-t copulas on heavy-tailed joint (pad_wind, shear_500m, gust)
rng = np.random.default_rng(99)
N_COP = 800
# True heavy-tailed 3D Student-t (df=3.5) with strong correlation
Sigma = np.array([[1.0, 0.78, 0.85], [0.78, 1.0, 0.65], [0.85, 0.65, 1.0]])
z_mv = rng.multivariate_normal(np.zeros(3), Sigma, size=N_COP)
scale_t = np.sqrt(rng.chisquare(3.5, size=(N_COP, 1)) / 3.5)
t_mv = z_mv / scale_t
data_cop = np.column_stack([
    np.clip(7.0 + 2.5 * t_mv[:, 0], 0.5, 30.0),   # wind_10m (m/s)
    np.clip(5.0 + 2.0 * t_mv[:, 1], 0.5, 25.0),   # shear_500m (m/s per km)
    np.clip(10.0 + 3.5 * t_mv[:, 2], 1.0, 40.0),  # gust_10m (m/s)
])
names_cop = ["wind_10m", "shear_500m", "gust_10m"]

cop_selected = Coupled(data_cop, names_cop, method=COPULA_METHOD, seed=42)
uncond_draws = cop_selected.draw(600, seed=1)
cond_cop = cop_selected.condition({"wind_10m": float(OBS_WIND_MS)})
cond_draws = cond_cop.draw(600, seed=2)

print(f"Selected copula: {COPULA_METHOD} (fitted nu={cop_selected.nu})")
print(f"shear_500m | Unconditioned median={np.median(uncond_draws[:,1]):.2f} (IQR={np.subtract(*np.percentile(uncond_draws[:,1],[75,25])):.2f})")
print(f"shear_500m | Conditioned on wind_10m={OBS_WIND_MS:.1f} m/s -> median={np.median(cond_draws[:,1]):.2f} (IQR={np.subtract(*np.percentile(cond_draws[:,1],[75,25])):.2f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].plot(leads, w_ens, lw=2.2, color="#0284c7", label="Ensemble weight (F → F2)")
axes[0].plot(leads, w_clim, lw=2.2, color="#7c3aed", label="Climate weight (M → C)")
axes[0].axvline(LEAD_DAYS, ls="--", color="#dc2626", lw=1.8, label=f"Selected: {LEAD_DAYS:+.0f} d (Mode {active_mode})")
axes[0].set(title="Lead-Time Router Cosine Blending (router.py)", xlabel="Forecast Lead Time (days)", ylabel="Blend Weight (0..1)")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].scatter(uncond_draws[:, 0], uncond_draws[:, 1], s=12, alpha=0.3, color="#64748b", label="Unconditioned prior")
axes[1].scatter(cond_draws[:, 0], cond_draws[:, 1], s=14, alpha=0.55, color="#ea580c", label=f"Conditioned (wind_10m={OBS_WIND_MS:.0f})")
axes[1].axvline(OBS_WIND_MS, ls="--", color="k", lw=1.2)
axes[1].set(title=f"Conditional {COPULA_METHOD} Draws (stats.Coupled)", xlabel="10 m Pad Wind (m/s)", ylabel="500 m Wind Shear (m/s/km)")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Left:** The smooth cosine ramps prevent sudden forecast discontinuities as a launch approaches from 46 days out into launch week.
- **Right:** Conditioning on an extreme pad wind (`OBS_WIND_MS = 14–18 m/s`) under `t_copula` shifts *and* widens the conditional shear distribution appropriately compared to a naive Gaussian copula—protecting launch safety on storm days.

---

## 4. Side-by-Side Gaussian vs. Student-$t$ Copula Tail Dependence

### Conceptual Intuition (Before the Math)
Why does `tempest_weather_v2.stats.Coupled` default to a **Student-$t$ copula** instead of a standard **Gaussian copula** when generating joint wind gusts across altitudes?

Think of two altitudes—**10 m pad gust** and **500 m max-Q gust**—with the *exact same linear correlation* $\rho = 0.70$:
- **Gaussian copula ("fair-weather friends"):** Even with $\rho = 0.70$, as you push further into the extreme upper tail ($u \to 1$), joint extremes **decouple completely**—its upper tail dependence coefficient is strictly **$\lambda_U = 0$** for any $\rho < 1$! That means a Gaussian copula dangerously under-predicts the chance of getting slammed by a simultaneous extreme gust at 10 m *and* 500 m on the same launch.
- **Student-$t$ copula (shared storm shock):** A Student-$t$ copula with low degrees of freedom $\nu$ multiplies *all* altitudes by a shared heavy-tailed random scale $\sqrt{\nu / \chi^2_\nu}$—just like a real cold front or convective outflow hitting the whole boundary layer at once! Its analytical upper tail dependence stays strictly positive:
$$\lambda_U(\rho, \nu) = 2\, t_{\nu + 1}\!\left(-\sqrt{\frac{(\nu + 1)(1 - \rho)}{1 + \rho}}\right) > 0$$

> **Predict → Run → Observe**
> - **Predict:** If both copulas share the exact same correlation (`CORR_RHO = 0.70`) and identical marginal distributions, which panel will pack **more red dots into the top-right 95% × 95% extreme-gust box**—Gaussian or Student-$t$? What happens to $\lambda_U$ as you increase `TRUE_NU` from `3` toward `30`?
> - **Run:** Execute Cell 4 below, then lower `TRUE_NU` to `2.5` or raise `TAIL_QUANTILE` to `0.98` and re-run.
> - **Observe:** Compare the empirical joint-extreme exceedance rate $P(U_{\text{500m}} > q \mid U_{\text{10m}} > q)$ in the left vs. middle panels, and trace on the right panel how Student-$t$'s $\lambda_U$ converges down toward the Gaussian $\lambda_U = 0$ floor as $\nu \to \infty$.

### Interactive Controls
Adjust `CORR_RHO`, `TRUE_NU` (degrees of freedom of the underlying storm data), and `TAIL_QUANTILE` ($q$, e.g. `0.95` for the top 5% gusts) to inspect side-by-side tail dependence.

In [ ]:
# @title 4. Side-by-Side Gaussian vs. Student-t Copula Tail-Dependence Comparison
CORR_RHO = 0.70 # @param {type:"slider", min:0.20, max:0.90, step:0.05}
TRUE_NU = 3.0 # @param {type:"slider", min:2.5, max:20.0, step:0.5}
TAIL_QUANTILE = 0.95 # @param {type:"slider", min:0.90, max:0.98, step:0.01}

from scipy import stats as sp_stats

def student_t_tail_dep(rho, nu):
    """Analytical upper/lower tail dependence lambda_U for a bivariate Student-t copula."""
    arg = -np.sqrt((nu + 1.0) * (1.0 - rho) / (1.0 + rho))
    return float(2.0 * sp_stats.t.cdf(arg, df=nu + 1.0))

def to_uniform_ranks(arr):
    """Convert 2D sample array (N, 2) to empirical uniform pseudo-observations in (0, 1)."""
    n = len(arr)
    u = np.empty_like(arr, dtype=float)
    for j in range(arr.shape[1]):
        order = np.argsort(arr[:, j])
        ranks = np.empty(n, dtype=float)
        ranks[order] = (np.arange(1, n + 1) - 0.5) / n
        u[:, j] = ranks
    return u

# Generate heavy-tailed training gusts at 10 m and 500 m AGL from a bivariate Student-t law
rng_td = np.random.default_rng(2026)
N_FIT, N_DRAW = 2000, 4000
cov_2d = np.array([[1.0, CORR_RHO], [CORR_RHO, 1.0]])
z_2d = rng_td.multivariate_normal([0.0, 0.0], cov_2d, size=N_FIT)
w_2d = np.sqrt(rng_td.chisquare(TRUE_NU, size=(N_FIT, 1)) / TRUE_NU)
t_2d = z_2d / w_2d
gust_train = np.column_stack([
    np.clip(11.0 + 3.0 * t_2d[:, 0], 1.0, 45.0),  # gust_10m (m/s)
    np.clip(15.0 + 4.0 * t_2d[:, 1], 1.0, 60.0),  # gust_500m (m/s)
])

# Fit BOTH Gaussian and Student-t copulas using tempest_weather_v2.stats.Coupled
cop_gauss = Coupled(gust_train, ["gust_10m", "gust_500m"], method="gaussian_copula", seed=7)
cop_t = Coupled(gust_train, ["gust_10m", "gust_500m"], method="t_copula", seed=7)

draw_g = cop_gauss.draw(N_DRAW, seed=101)
draw_t = cop_t.draw(N_DRAW, seed=101)
u_g = to_uniform_ranks(draw_g)
u_t = to_uniform_ranks(draw_t)

# Empirical conditional tail exceedance P(U_500m > q | U_10m > q)
q = float(TAIL_QUANTILE)
mask_g_hi = (u_g[:, 0] > q) & (u_g[:, 1] > q)
mask_t_hi = (u_t[:, 0] > q) & (u_t[:, 1] > q)
p_cond_g = float(mask_g_hi.sum() / max((u_g[:, 0] > q).sum(), 1))
p_cond_t = float(mask_t_hi.sum() / max((u_t[:, 0] > q).sum(), 1))
nu_fit = float(cop_t.nu)
lam_t_theory = student_t_tail_dep(CORR_RHO, nu_fit)

print(f"Fitted Student-t degrees of freedom: nu = {nu_fit:.1f} (data true nu = {TRUE_NU:.1f}, rho = {CORR_RHO:.2f})")
print(f"Analytical upper tail dependence lambda_U | Gaussian: 0.000 | Student-t (nu={nu_fit:.1f}): {lam_t_theory:.3f}")
print(f"Empirical P(gust_500m > Q{int(100*q)} | gust_10m > Q{int(100*q)}) | Gaussian: {100*p_cond_g:.1f}% ({mask_g_hi.sum()} pts) | Student-t: {100*p_cond_t:.1f}% ({mask_t_hi.sum()} pts)")

# Plot side-by-side Gaussian vs. Student-t uniform copula draws + analytical lambda_U(nu) curve
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.5))
for ax, u_pts, mask_hi, title, p_cond, lam_lbl in [
    (axes[0], u_g, mask_g_hi, f"Gaussian Copula (λ_U = 0)\nP(U₂>{q:.2f} | U₁>{q:.2f}) = {100*p_cond_g:.1f}%", p_cond_g, "Zero tail dependence"),
    (axes[1], u_t, mask_t_hi, f"Student-t Copula (fitted ν={nu_fit:.1f}, λ_U={lam_t_theory:.2f})\nP(U₂>{q:.2f} | U₁>{q:.2f}) = {100*p_cond_t:.1f}%", p_cond_t, "Co-occurring extreme gusts"),
]:
    ax.scatter(u_pts[~mask_hi, 0], u_pts[~mask_hi, 1], s=6, alpha=0.22, color="#64748b", rasterized=True)
    ax.scatter(u_pts[mask_hi, 0], u_pts[mask_hi, 1], s=14, alpha=0.85, color="#dc2626", label=f"Joint extremes ({mask_hi.sum()} pts)")
    ax.add_patch(plt.Rectangle((q, q), 1.0 - q, 1.0 - q, fill=True, facecolor="#fecaca", alpha=0.35, edgecolor="#b91c1c", lw=1.5))
    ax.set(title=title, xlabel="10 m Gust Quantile U₁", ylabel="500 m Gust Quantile U₂", xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.25)

nu_grid = np.linspace(2.0, 30.0, 200)
lam_curve = [student_t_tail_dep(CORR_RHO, float(v)) for v in nu_grid]
axes[2].plot(nu_grid, lam_curve, lw=2.4, color="#2563eb", label=f"Student-t λ_U(ρ={CORR_RHO:.2f}, ν)")
axes[2].axhline(0.0, ls="--", lw=1.8, color="#dc2626", label="Gaussian copula (λ_U = 0)")
axes[2].scatter([nu_fit], [lam_t_theory], s=75, color="#ea580c", zorder=5, label=f"Fitted ν={nu_fit:.1f} (λ_U={lam_t_theory:.2f})")
axes[2].set(title="Analytical Upper Tail Dependence vs. ν", xlabel="Degrees of Freedom ν", ylabel="Upper Tail Dependence λ_U")
axes[2].set_ylim(-0.03, max(lam_curve) * 1.15)
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**How to interpret the plot:**
- **Left vs. Middle:** Both copulas reproduce the exact same marginal gust distributions and central correlation, yet the **Student-$t$ copula** clusters significantly more simultaneous extremes inside the red top-right box ($U_1 > q, U_2 > q$)—preserving multi-altitude storm gusts that the Gaussian copula misses.
- **Right:** As degrees of freedom $\nu$ grow large, the Student-$t$ copula approaches the Gaussian limit ($\lambda_U \to 0$); keeping a fitted finite $\nu$ ensures Tempest-ARKNET does not underestimate simultaneous structural loads across altitudes.